In [2]:
import os

BASE_INPUT = '/kaggle/input/datasets'

datasets = {
    'drozy':   f'{BASE_INPUT}/ahmedfaroukksiu/drozy',
    'nthu':    f'{BASE_INPUT}/ikhlaselhamly/nthu-ddd',
    'yawdd':   f'{BASE_INPUT}/enider/yawdd-dataset',
    'cew':     f'{BASE_INPUT}/faisal7/cew-dataset',
    'mrl':     f'{BASE_INPUT}/rameezakther/mrl-eye-open-or-close-dataset',
    'nitymed': f'{BASE_INPUT}/nikospetrellis/nitymed',
}

BASE = '/kaggle/working/data'
for ds in ['drozy', 'yawdd', 'nitymed']:
    os.makedirs(f'{BASE}/{ds}/frames', exist_ok=True)

print('Dataset Mount Check')
for name, path in datasets.items():
    if os.path.exists(path):
        size = sum(
            os.path.getsize(os.path.join(dp, f))
            for dp, _, files in os.walk(path)
            for f in files
        )
        print(f'✅ {name.upper():10} | {size/1e9:.2f} GB | {path}')
    else:
        print(f'❌ {name.upper():10} | NOT FOUND')

Dataset Mount Check
✅ DROZY      | 3.38 GB | /kaggle/input/datasets/ahmedfaroukksiu/drozy
✅ NTHU       | 0.82 GB | /kaggle/input/datasets/ikhlaselhamly/nthu-ddd
✅ YAWDD      | 5.48 GB | /kaggle/input/datasets/enider/yawdd-dataset
✅ CEW        | 0.00 GB | /kaggle/input/datasets/faisal7/cew-dataset
✅ MRL        | 0.08 GB | /kaggle/input/datasets/rameezakther/mrl-eye-open-or-close-dataset
✅ NITYMED    | 0.70 GB | /kaggle/input/datasets/nikospetrellis/nitymed


In [3]:
import cv2, os, shutil
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm

def extract_frames(video_path, out_dir, label, fps_sample=5, max_frames=500):
    vid_name = os.path.splitext(os.path.basename(video_path))[0]
    save_dir = os.path.join(out_dir, label, vid_name)
    os.makedirs(save_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    interval = max(1, int(fps / fps_sample))
    frame_id, saved = 0, 0
    while saved < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_id % interval == 0:
            cv2.imwrite(f'{save_dir}/f{saved:05d}.jpg', frame)
            saved += 1
        frame_id += 1
    cap.release()

def split_videos(records):
    labels = [r['label'] for r in records]
    train, temp = train_test_split(records, test_size=0.30, stratify=labels, random_state=42)
    val, test   = train_test_split(temp, test_size=0.50,
                                   stratify=[r['label'] for r in temp], random_state=42)
    return train, val, test

def split_videos_single_class(records):
    train, temp = train_test_split(records, test_size=0.30, random_state=42)
    val, test   = train_test_split(temp, test_size=0.50, random_state=42)
    return train, val, test

def collect_videos(folder, label):
    exts = {'.avi', '.mp4', '.mkv', '.mov'}
    return [
        {'path': os.path.join(r, f), 'label': label}
        for r, _, files in os.walk(folder)
        for f in files if Path(f).suffix.lower() in exts
    ]

def run_extraction(split_records_map, base_out):
    for split, records in split_records_map.items():
        for r in tqdm(records, desc=f'{base_out.split("/")[-1]}/{split}'):
            extract_frames(r['path'], f'{base_out}/{split}', r['label'])

# ── Clear old frames ──
for ds in ['yawdd', 'drozy', 'nitymed']:
    p = f'{BASE}/{ds}/frames'
    if os.path.exists(p):
        shutil.rmtree(p)
        print(f'Cleared {p}')

# ── YawDD ──
YAWDD_LABEL_MAP = {'Yawning': 'drowsy', 'Normal': 'alert', 'Talking': 'alert'}
yawdd_records = []
for gender in ['Female_mirror', 'Male_mirror Avi Videos']:
    folder = f'{BASE_INPUT}/enider/yawdd-dataset/Mirror/Mirror/{gender}'
    if not os.path.exists(folder):
        print(f'Not found: {folder}')
        continue
    for fname in os.listdir(folder):
        if not fname.endswith('.avi'):
            continue
        label = next((v for k, v in YAWDD_LABEL_MAP.items() if k in fname), None)
        if label:
            yawdd_records.append({'path': os.path.join(folder, fname), 'label': label})

train_v, val_v, test_v = split_videos(yawdd_records)
print(f'YawDD — train:{len(train_v)} val:{len(val_v)} test:{len(test_v)}')
run_extraction({'train': train_v, 'val': val_v, 'test': test_v},
               f'{BASE}/yawdd/frames')

# ── DROZY ──
drozy_records = []
drozy_vid_dir = f'{BASE_INPUT}/ahmedfaroukksiu/drozy/DROZY/videos_i8'
for vid in os.listdir(drozy_vid_dir):
    if not vid.endswith(('.avi', '.mp4', '.mkv')):
        continue
    try:
        session = int(vid.split('-')[1].split('.')[0])
    except:
        print(f'Skipped: {vid}')
        continue
    label = 'alert' if session == 1 else 'drowsy'
    drozy_records.append({'path': os.path.join(drozy_vid_dir, vid), 'label': label})

train_v, val_v, test_v = split_videos(drozy_records)
print(f'DROZY — train:{len(train_v)} val:{len(val_v)} test:{len(test_v)}')
run_extraction({'train': train_v, 'val': val_v, 'test': test_v},
               f'{BASE}/drozy/frames')

# ── NITYMED ──
nitymed_records = collect_videos(
    f'{BASE_INPUT}/nikospetrellis/nitymed/DSM_Dataset-HDTV720', 'drowsy'
)
train_v, val_v, test_v = split_videos_single_class(nitymed_records)
print(f'NITYMED — train:{len(train_v)} val:{len(val_v)} test:{len(test_v)}')
run_extraction({'train': train_v, 'val': val_v, 'test': test_v},
               f'{BASE}/nitymed/frames')

# ── Summary ──
print('\n===== Frame Extraction Summary =====')
for ds in ['yawdd', 'drozy', 'nitymed']:
    for split in ['train', 'val', 'test']:
        for cls in ['alert', 'drowsy']:
            p = f'{BASE}/{ds}/frames/{split}/{cls}'
            if not os.path.exists(p):
                continue
            n_vids   = len(os.listdir(p))
            n_frames = sum(
                len(os.listdir(os.path.join(p, v)))
                for v in os.listdir(p)
                if os.path.isdir(os.path.join(p, v))
            )
            print(f'  {ds:8} | {split:5} | {cls:6} | {n_vids:3} videos | {n_frames:6} frames')

Cleared /kaggle/working/data/yawdd/frames
Cleared /kaggle/working/data/drozy/frames
Cleared /kaggle/working/data/nitymed/frames
YawDD — train:223 val:48 test:48


frames/test: 100%|██████████| 48/48 [00:31<00:00,  1.52it/s]


DROZY — train:25 val:5 test:6


frames/test: 100%|██████████| 6/6 [00:08<00:00,  1.40s/it]


NITYMED — train:88 val:19 test:19


frames/test: 100%|██████████| 19/19 [00:48<00:00,  2.54s/it]


===== Frame Extraction Summary =====
  yawdd    | train | alert  | 144 videos |  22447 frames
  yawdd    | train | drowsy |  79 videos |   9234 frames
  yawdd    | val   | alert  |  31 videos |   4556 frames
  yawdd    | val   | drowsy |  17 videos |   1829 frames
  yawdd    | test  | alert  |  31 videos |   4614 frames
  yawdd    | test  | drowsy |  17 videos |   2037 frames
  drozy    | train | alert  |   8 videos |   4000 frames
  drozy    | train | drowsy |  17 videos |   8500 frames
  drozy    | val   | alert  |   2 videos |   1000 frames
  drozy    | val   | drowsy |   3 videos |   1500 frames
  drozy    | test  | alert  |   2 videos |   1000 frames
  drozy    | test  | drowsy |   4 videos |   2000 frames
  nitymed  | train | drowsy |  88 videos |  17844 frames
  nitymed  | val   | drowsy |  19 videos |   4128 frames
  nitymed  | test  | drowsy |  19 videos |   4365 frames


In [5]:
import cv2, os, gc
import numpy as np
from PIL import Image
from pathlib import Path
from tqdm import tqdm

# ── Initialize extractor first ──
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import urllib.request

model_path = '/kaggle/working/face_landmarker.task'
if not os.path.exists(model_path):
    print('Downloading face landmarker model...')
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task',
        model_path
    )

class FaceRegionExtractor:
    LEFT_EYE  = [33, 160, 158, 133, 153, 144]
    RIGHT_EYE = [362, 385, 387, 263, 373, 380]
    MOUTH     = [61, 291, 39, 181, 0, 17, 269, 405]

    def __init__(self):
        base_options  = python.BaseOptions(model_asset_path=model_path)
        options       = vision.FaceLandmarkerOptions(
            base_options=base_options,
            num_faces=1,
            min_face_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )
        self.detector = vision.FaceLandmarker.create_from_options(options)

    def _crop_region(self, img, landmarks, indices, pad=0.3):
        h, w = img.shape[:2]
        pts  = [(int(landmarks[i].x * w), int(landmarks[i].y * h)) for i in indices]
        x1   = max(0, min(p[0] for p in pts) - int(w * pad))
        x2   = min(w, max(p[0] for p in pts) + int(w * pad))
        y1   = max(0, min(p[1] for p in pts) - int(h * pad))
        y2   = min(h, max(p[1] for p in pts) + int(h * pad))
        crop = img[y1:y2, x1:x2]
        return crop if crop.size > 0 else img

    def extract(self, img_rgb):
        import mediapipe as mp
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
        results  = self.detector.detect(mp_image)
        if not results.face_landmarks:
            return img_rgb, img_rgb, img_rgb
        lms  = results.face_landmarks[0]
        h, w = img_rgb.shape[:2]
        xs   = [lm.x * w for lm in lms]
        ys   = [lm.y * h for lm in lms]
        x1   = max(0, int(min(xs)) - 20)
        x2   = min(w, int(max(xs)) + 20)
        y1   = max(0, int(min(ys)) - 20)
        y2   = min(h, int(max(ys)) + 20)
        face = img_rgb[y1:y2, x1:x2] if (y2 > y1 and x2 > x1) else img_rgb
        eyes  = self._crop_region(img_rgb, lms, self.LEFT_EYE + self.RIGHT_EYE, pad=0.15)
        mouth = self._crop_region(img_rgb, lms, self.MOUTH, pad=0.2)
        return face, eyes, mouth

extractor = FaceRegionExtractor()
print('Extractor ready')

# ── rest of pre-extraction code below (unchanged) ──
REGIONS_BASE = '/kaggle/working/regions'

def preextract_regions(src_root, dst_root):
    img_exts = {'.jpg', '.jpeg', '.png', '.bmp'}
    all_files = [
        os.path.join(r, f)
        for r, _, files in os.walk(src_root)
        for f in files if Path(f).suffix.lower() in img_exts
    ]
    print(f'Pre-extracting {len(all_files)} frames from {src_root}...')
    for src_path in tqdm(all_files):
        rel        = os.path.relpath(src_path, src_root)
        stem       = os.path.splitext(rel)[0]
        face_path  = os.path.join(dst_root, 'face',  stem + '.jpg')
        eye_path   = os.path.join(dst_root, 'eye',   stem + '.jpg')
        mouth_path = os.path.join(dst_root, 'mouth', stem + '.jpg')
        if all(os.path.exists(p) for p in [face_path, eye_path, mouth_path]):
            continue
        os.makedirs(os.path.dirname(face_path),  exist_ok=True)
        os.makedirs(os.path.dirname(eye_path),   exist_ok=True)
        os.makedirs(os.path.dirname(mouth_path), exist_ok=True)
        try:
            img = np.array(Image.open(src_path).convert('RGB'))
        except:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        face, eye, mouth = extractor.extract(img)
        def save(region, path):
            region = cv2.resize(region, (224, 224))
            cv2.imwrite(path, cv2.cvtColor(region, cv2.COLOR_RGB2BGR))
        save(face,  face_path)
        save(eye,   eye_path)
        save(mouth, mouth_path)
    print(f'Done → {dst_root}')
    gc.collect()

# ── Video frames ──
for ds in ['yawdd', 'drozy', 'nitymed']:
    src = f'{BASE}/{ds}/frames'
    dst = f'{REGIONS_BASE}/{ds}'
    if os.path.exists(src):
        preextract_regions(src, dst)

# ── Image datasets ──
img_sources = {
    'mrl_open_train':  f'{BASE_INPUT}/rameezakther/mrl-eye-open-or-close-dataset/mrl_dataset/train/open_eyes_sample',
    'mrl_close_train': f'{BASE_INPUT}/rameezakther/mrl-eye-open-or-close-dataset/mrl_dataset/train/close_eyes_sample',
    'mrl_open_test':   f'{BASE_INPUT}/rameezakther/mrl-eye-open-or-close-dataset/mrl_dataset/test/open_eyes_test',
    'mrl_close_test':  f'{BASE_INPUT}/rameezakther/mrl-eye-open-or-close-dataset/mrl_dataset/test/close_eyes_test',
    'nthu_notdrowsy':  f'{BASE_INPUT}/ikhlaselhamly/nthu-ddd/NTHU-DDD/notdrowsy',
    'nthu_drowsy':     f'{BASE_INPUT}/ikhlaselhamly/nthu-ddd/NTHU-DDD/drowsy',
    'cew_open':        f'{BASE_INPUT}/faisal7/cew-dataset/dataset_B_Eye_Images/dataset_B_Eye_Images/openEyes',
    'cew_closed':      f'{BASE_INPUT}/faisal7/cew-dataset/dataset_B_Eye_Images/dataset_B_Eye_Images/closedEyes',
}
for name, src in img_sources.items():
    if os.path.exists(src):
        preextract_regions(src, f'{REGIONS_BASE}/images/{name}')

print('\n✅ All regions pre-extracted.')

2026-05-02 14:24:54.735127: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777731894.962678      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777731895.029218      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777731895.516779      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777731895.516818      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777731895.516821      57 computation_placer.cc:177] computation placer alr

Extractor ready


W0000 00:00:1777731914.486171    2210 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1777731914.526512    2213 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1777731914.548964    2215 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Pre-extracting 44717 frames from /kaggle/working/data/yawdd/frames...


100%|██████████| 44717/44717 [15:49<00:00, 47.08it/s]


...

Done → /kaggle/working/regions/images/nthu_drowsy
Pre-extracting 2462 frames from /kaggle/input/datasets/faisal7/cew-dataset/dataset_B_Eye_Images/dataset_B_Eye_Images/openEyes...


100%|██████████| 2462/2462 [00:29<00:00, 83.13it/s]


Done → /kaggle/working/regions/images/cew_open
Pre-extracting 2384 frames from /kaggle/input/datasets/faisal7/cew-dataset/dataset_B_Eye_Images/dataset_B_Eye_Images/closedEyes...


100%|██████████| 2384/2384 [00:25<00:00, 94.03it/s] 

Done → /kaggle/working/regions/images/cew_closed

✅ All regions pre-extracted.


In [6]:
import shutil

for ds in ['yawdd', 'drozy', 'nitymed']:
    p = f'{BASE}/{ds}/frames'
    if os.path.exists(p):
        shutil.rmtree(p)
        print(f'Deleted {p}')

# Verify space freed
import subprocess
result = subprocess.run(['du', '-sh', '/kaggle/working'], capture_output=True, text=True)
print(f'Working dir size: {result.stdout}')

Deleted /kaggle/working/data/yawdd/frames
Deleted /kaggle/working/data/drozy/frames
Deleted /kaggle/working/data/nitymed/frames
Working dir size: 5.1G	/kaggle/working

